In [4]:
import requests
import pandas as pd
import pymysql

from datetime import datetime
from dateutil.relativedelta import relativedelta


# --------------------------------
# 1. USGS API
# --------------------------------

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"


# --------------------------------
# 2. 5-year date range
# --------------------------------

end_date = datetime.now()
start_date = end_date - relativedelta(years=5)

current_date = start_date


# --------------------------------
# 3. MySQL connection
# --------------------------------

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="1234"
)

cursor = conn.cursor() 

cursor.execute("CREATE DATABASE IF NOT EXISTS earthquakesss") 

conn.commit() 

conn.close() 


conn = pymysql.connect(
    host="localhost",
    user="root",
    password="1234",
    database="earthquakesss"
)

cursor = conn.cursor()


# --------------------------------
# 4. SQL INSERT
# --------------------------------

create_table = """
CREATE TABLE IF NOT EXISTS earthquakes (
    id INT AUTO_INCREMENT PRIMARY KEY,
    mag FLOAT,
    place VARCHAR(255),
    time DATETIME,
    magType VARCHAR(50),
    longitude FLOAT,
    latitude FLOAT,
    depth FLOAT
)
"""

cursor.execute(create_table)
conn.commit()  

sql = """
INSERT INTO earthquakes
(mag, place, time, magType, longitude, latitude, depth)
VALUES (%s, %s, %s, %s, %s, %s, %s)
"""


# --------------------------------
# 5. Counters
# --------------------------------

total_inserted = 0


# --------------------------------
# 6. Download month by month
# --------------------------------

while current_date < end_date:

    next_date = current_date + relativedelta(months=1)

    if next_date > end_date:
        next_date = end_date

    print()
    print("--------------------------------")
    print("Downloading:", current_date.date(), "to", next_date.date())
    print("--------------------------------")


    offset = 1


    # --------------------------------
    # 7. Handle pagination
    # --------------------------------

    while True:

        parameters = {
            "format": "geojson",
            "starttime": current_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "endtime": next_date.strftime("%Y-%m-%dT%H:%M:%S"),
            "limit": 20000,
            "offset": offset,
            "orderby": "time-asc"
        }


        response = requests.get(
            url,
            params=parameters,
            timeout=60
        )


        print("Status:", response.status_code)

        response.raise_for_status()

        data = response.json()

        features = data["features"]

        print("Received:", len(features))


        # --------------------------------
        # 8. Stop if no data
        # --------------------------------

        if len(features) == 0:
            break


        # --------------------------------
        # 9. Create MySQL rows
        # --------------------------------

        rows = []


        for earthquake in features:

            properties = earthquake["properties"]
            coordinates = earthquake["geometry"]["coordinates"]


            row = (
                properties["mag"],
                properties["place"],

                pd.to_datetime(
                    properties["time"],
                    unit="ms"
                ).to_pydatetime(),

                properties["magType"],

                coordinates[0],
                coordinates[1],
                coordinates[2]
            )


            rows.append(row)


        # --------------------------------
        # 10. Insert into MySQL
        # --------------------------------

        cursor.executemany(
            sql,
            rows
        )

        conn.commit()


        total_inserted += len(rows)


        print("Inserted:", len(rows))
        print("Total so far:", total_inserted)


        # --------------------------------
        # 11. Pagination
        # --------------------------------

        if len(features) < 20000:
            break

        offset += 20000


    # --------------------------------
    # 12. Move to next month
    # --------------------------------

    current_date = next_date


# --------------------------------
# 13. Finished
# --------------------------------

print()
print("====================================")
print("5-YEAR DOWNLOAD COMPLETE")
print("Total rows inserted:", total_inserted)
print("====================================")


cursor.close()
conn.close()


--------------------------------
Downloading: 2021-10-06 to 2021-11-06
--------------------------------
Status: 200
Received: 12115
Inserted: 12115
Total so far: 12115

--------------------------------
Downloading: 2021-11-06 to 2021-12-06
--------------------------------
Status: 200
Received: 11489
Inserted: 11489
Total so far: 23604

--------------------------------
Downloading: 2021-12-06 to 2022-01-06
--------------------------------
Status: 200
Received: 11210
Inserted: 11210
Total so far: 34814

--------------------------------
Downloading: 2022-01-06 to 2022-02-06
--------------------------------
Status: 200
Received: 13452
Inserted: 13452
Total so far: 48266

--------------------------------
Downloading: 2022-02-06 to 2022-03-06
--------------------------------
Status: 200
Received: 11653
Inserted: 11653
Total so far: 59919

--------------------------------
Downloading: 2022-03-06 to 2022-04-06
--------------------------------
Status: 200
Received: 12676
Inserted: 12676
Total 

In [5]:
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="1234",
    database="earthquakesss"
)

cursor = conn.cursor()

cursor.execute("SELECT COUNT(*) FROM earthquakes")

print(cursor.fetchone())

(739532,)
